# Experiment 10: Text Classification (Reuters News)

Pipeline: **clean text -> TF-IDF features -> train classifier -> evaluate**

- **TF-IDF** = word count x log(N / docs containing word). Rare-but-present words weigh more.
- **Naive Bayes:** fast probabilistic baseline. **Linear SVM:** max-margin line. **Logistic Regression:** probabilities + readable weights.
- **Macro-F1** treats every class equally, so it shows weak small classes that accuracy hides.

In [ ]:
# !pip install nltk scikit-learn matplotlib

In [ ]:
import re
import nltk
import matplotlib.pyplot as plt
from nltk.corpus import reuters, stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import make_pipeline
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, ConfusionMatrixDisplay

nltk.download('reuters', quiet=True)
nltk.download('stopwords', quiet=True)
stop = set(stopwords.words('english'))

## Load 6 topics (single-label documents, official train/test split)

In [ ]:
TOPICS = ['earn', 'acq', 'crude', 'trade', 'money-fx', 'interest']
ids = [f for f in reuters.fileids() if len(reuters.categories(f)) == 1 and reuters.categories(f)[0] in TOPICS]

def clean(text):
    text = re.sub(r'[^a-z\s]', ' ', text.lower())          # letters only
    return ' '.join(w for w in text.split() if w not in stop and len(w) > 2)

X_train = [clean(reuters.raw(f)) for f in ids if f.startswith('training/')]
y_train = [reuters.categories(f)[0] for f in ids if f.startswith('training/')]
X_test = [clean(reuters.raw(f)) for f in ids if f.startswith('test/')]
y_test = [reuters.categories(f)[0] for f in ids if f.startswith('test/')]

print('Train:', len(X_train), '| Test:', len(X_test))
for t in TOPICS:
    print(f'{t:9} {y_train.count(t)}')        # very imbalanced: earn is about half

## Train and compare three classifiers

In [ ]:
models = {
    'Naive Bayes': MultinomialNB(alpha=0.1),
    'Linear SVM': LinearSVC(),
    'Logistic Regression': LogisticRegression(max_iter=2000, C=10),
}

for name, clf in models.items():
    # pipeline fits TF-IDF on training data only
    pipe = make_pipeline(TfidfVectorizer(min_df=2, ngram_range=(1, 2), sublinear_tf=True), clf)
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    models[name] = pipe
    print(f'{name:20} accuracy={accuracy_score(y_test, pred):.3f}  macro-F1={f1_score(y_test, pred, average="macro"):.3f}')

## Detailed report and confusion matrix (Linear SVM)

In [ ]:
best = models['Linear SVM']
pred = best.predict(X_test)
print(classification_report(y_test, pred, digits=3))

ConfusionMatrixDisplay.from_predictions(y_test, pred, labels=TOPICS, xticks_rotation=45)
plt.title('Confusion matrix: Linear SVM'); plt.show()

## Top words per topic (Logistic Regression weights)

In [ ]:
lr = models['Logistic Regression']
words = lr[0].get_feature_names_out()
clf = lr[-1]
for topic, weights in zip(clf.classes_, clf.coef_):
    print(f'{topic:9}', ', '.join(words[i] for i in weights.argsort()[::-1][:8]))

## Try it on new text

In [ ]:
for text in ['Oil prices rose after OPEC cut crude production',
             'The company agreed to buy its rival for 2 billion dollars',
             'The central bank raised interest rates by half a point']:
    print(best.predict([clean(text)])[0], '<-', text)